# Function Calling (Tool Use)

## Task Description
Demonstrate **Function Calling** (also known as Tool Use), where an LLM generates structured arguments to call external functions instead of just generating text.

**Focus:** This notebook uses `Qwen/Qwen2.5-1.5B-Instruct`, a small but powerful model capable of tool use.

In this notebook, we will:
1. **Install:** Install `transformers`.
2. **Define Tools:** Define a Python function (e.g., `get_current_weather`) and its schema.
3. **Load Model:** Load the Qwen model.
4. **Inference:** Pass the user query and tool definition to the model.
5. **Parse:** Extract the tool call arguments from the model output.

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source models and is optimized to run on the Google Colab Free Tier.

## 1. Installation


In [ ]:
!pip install -q --upgrade transformers torch accelerate

## 2. Imports


In [ ]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

## 3. Define Tools
We define the tool schema (JSON format) and the actual function.

In [ ]:
# Actual function
def get_current_weather(location, unit="celsius"):
    """Get the current weather in a given location"""
    if "london" in location.lower():
        return json.dumps({"location": location, "temperature": "20", "unit": unit, "description": "Cloudy"})
    elif "paris" in location.lower():
        return json.dumps({"location": location, "temperature": "22", "unit": unit, "description": "Sunny"})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

# Tool Definition (Schema)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

## 3. Load Data
User query.

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the weather like in London today?"}
]

## 5. Load Model
Load `Qwen/Qwen2.5-1.5B-Instruct`.

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

print("Model loaded.")

## 6. Inference
We use `apply_chat_template` to format the tools and messages.

In [ ]:
# Format inputs
text = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    add_generation_prompt=True,
    tokenize=False
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate
outputs = model.generate(**inputs, max_new_tokens=512)
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# We need to extract the tool call. For simplicity, we print the raw output.
# In a real app, you would parse the tool call (Qwen uses XML-like tags or specific JSON formats depending on the version/template).
print("Full Output:")
print(output_text)

## 7. Save Results


In [ ]:
with open('results.txt', 'w') as f:
    f.write(output_text)
print('Results saved to results.txt')

# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
     rate = save_rate if 'save_rate' in locals() else 16000
     try:
         wavfile.write('generated_audio.wav', rate, audio_data)
         print("Saved generated_audio.wav")
     except:
         pass


## 8. Advanced Usage


**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
# Execute the tool call
# In a real agent loop, you would parse 'output_text', execute 'get_current_weather("London")', 
# and pass the result back to the model.
print("Next step: Parse the output and execute the function!")